# SOXX 已披露篮子下跌贡献集中度：训练来源预检

仅检查首个训练日的原始证券身份和价格源目录。

In [1]:
import hashlib
import json
from pathlib import Path

import qlib
from qlib.data import D
from src.data.minute_intake import atomic_json

root = Path.cwd().resolve()
card = root / 'research/stage1_disclosed_loss_concentration_preflight_20261023'
plan = json.loads((card / 'plan.json').read_text())
source = plan['source']
def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
for name, digest_key in [
    ('first_original_report', 'first_original_report_sha256'),
    ('first_original_report_transport_receipt', 'first_original_report_transport_receipt_sha256'),
    ('pit_price_provider_manifest', 'pit_price_provider_manifest_sha256'),
    ('pit_price_coverage_manifest', 'pit_price_coverage_manifest_sha256'),
    ('current_basket_provider_manifest', 'current_basket_provider_manifest_sha256'),
]:
    assert sha(root / source[name]) == source[digest_key], name
report = json.loads((root / source['first_original_report']).read_text())
transport = json.loads((root / source['first_original_report_transport_receipt']).read_text())
coverage_manifest = json.loads((root / source['pit_price_coverage_manifest']).read_text())
assert transport['nas_scoped_restore_verified'] and transport['nas_evidence']['sha256'] == source['first_original_report_NAS_sha256']
provider = json.loads((root / source['pit_price_provider_manifest']).read_text())
current = json.loads((root / source['current_basket_provider_manifest']).read_text())
assert report['source_NAS_archive_sha256'] == source['first_original_report_NAS_sha256']
assert report['parsed_equity_rows'] == 30 and all(report['checks'].values())
assert report['accepted_at_utc'] == '2010-12-30T22:51:52.000Z'
qlib.init(provider_uri=str(root / 'data/qlib/soxx_pit_leveraged_training_v1'), region='us')
first = D.calendar(start_time='2011-01-03', end_time='2011-01-03', freq='day')
assert len(first) == 1 and str(first[0].date()) == '2011-01-03'
names = ['SanDisk Corp.', 'Altera Corp.', 'Broadcom Corp. Class A']
selected = []
for name in names:
    matches = [row for row in report['equities'] if row['security_original'].startswith(name)]
    assert len(matches) == 1, name
    selected.append(matches[0])
assert any(row['security_original'].startswith('Avago Technologies') for row in report['equities'])
legacy_labels = ['SNDK', 'ALTR', 'BRCM']
assert all(label not in provider['instruments'] and label not in current['instruments'] for label in legacy_labels)
assert all(label not in coverage_manifest['price_hashes'] for label in legacy_labels)
assert 'Yahoo does not retain every delisted ticker; unavailable historical members are excluded and weight coverage is reported.' in coverage_manifest['limitations']
combined = sum(row['reported_equity_weight'] for row in selected)
assert 0.16 < combined < 0.18
raw_paths = [Path(x) for x in str(provider['source']).split(';')]
missing_raw_sources = [str(x) for x in raw_paths if not x.exists()]
result = {
    'schema_version': 1,
    'status': 'needs_original_security_price_identity_and_history_no_fit',
    'plan_sha256': sha(card / 'plan.json'),
    'first_report_date': report['report_date'],
    'first_report_accepted_at_utc': report['accepted_at_utc'],
    'first_train_signal': '2011-01-03',
    'first_train_signal_in_native_Qlib_calendar': True,
    'original_report_equity_rows': report['parsed_equity_rows'],
    'three_required_security_names': [row['security_original'] for row in selected],
    'three_required_disclosed_weights': [row['reported_equity_weight'] for row in selected],
    'three_required_combined_disclosed_weight': combined,
    'distinct_original_Avago_and_Broadcom_in_same_report': True,
    'legacy_symbol_labels_absent_from_both_provider_inventories': legacy_labels,
    'provider_instrument_count': len(provider['instruments']),
    'current_basket_instrument_count': len(current['instruments']),
    'certified_original_security_to_provider_mapping_present': False,
    'referenced_raw_provider_sources_missing_locally': missing_raw_sources,
    'report_source_NAS_sha256': report['source_NAS_archive_sha256'],
    'source_price_provider_manifest_sha256': source['pit_price_provider_manifest_sha256'],
    'source_price_coverage_manifest_sha256': source['pit_price_coverage_manifest_sha256'],
    'original_report_NAS_scoped_restore_verified': transport['nas_scoped_restore_verified'],
    'full_2011_2022_feature_source_qualified': False,
    'market_outcome_reads': 0,
    'new_HTTP_GETs': 0,
    'model_fits': 0,
    'native_accounts': 0,
    'interpretation': 'The first Stage1 training basket includes three separate historical issuers totaling about 16.9% of disclosed equity value; the existing Qlib inventories do not certify separate original-security price series for them. This is a source gap, not evidence that the proposed factor predicts SOXX.',
}
atomic_json(result, card / 'source_preflight_result.json')
summary = {k: result[k] for k in ['status', 'plan_sha256', 'first_train_signal', 'three_required_combined_disclosed_weight', 'full_2011_2022_feature_source_qualified', 'market_outcome_reads', 'model_fits', 'native_accounts']}
summary.update(experiment=plan['experiment'], decision='source_gap_no_feature_or_fit', objective_achieved=False, source_result='source_preflight_result.json', artifact_policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]})
atomic_json(summary, card / 'summary.json')
(card / 'conclusion.md').write_text(
    '# 已披露篮子下跌贡献集中度：训练来源门\n\n'
    '该机制拟区分“少数权重股拖累”与“全篮子广泛下跌”，不同于旧静态权重分散度和上涨广度。此卡只检查首个训练日来源，未计算因子或未来收益。\n\n'
    f'2010-10-31 原始 SOXX 30 股报告于 {report["accepted_at_utc"]} 被 SEC 接收，先于 Qlib 2011-01-03 首个训练日。SanDisk、Altera、原 Broadcom 三只**不同证券**合计占已披露普通股市值 {combined:.2%}；同一报告还单独列有 Avago，不能用一个后来的 AVGO 序列合并两者。现有两个 Qlib 成分价格清单都不含 SNDK/ALTR/BRCM 标签，也没有逐只旧证券到价格序列的已认证映射。故当前来源不能支持完整的 2011–2022 Stage1 训练特征。\n\n'
    '这只是已有提供器的身份/覆盖缺口，不断言市场上不存在历史价格。旧来源清单另引用若干本机已缺失的原始清单文件；保留 Qlib 对象，不把它当可丢弃缓存。下一步先找可哈希、可恢复并核对公司行动的旧证券价格源与映射；在此之前不拟合、不建账户、不把缺失成员重归一化。0 HTTP、0 市场收益读取、0 拟合、0 账户，SOXX 总目标未达成。\n'
)
print(result['status'], result['three_required_combined_disclosed_weight'], result['referenced_raw_provider_sources_missing_locally'])


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[73057:MainThread](2026-10-23 03:43:39,294) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[73057:MainThread](2026-10-23 03:43:39,499) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[73057:MainThread](2026-10-23 03:43:39,499) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxx_pit_leveraged_training_v1')}


needs_original_security_price_identity_and_history_no_fit 0.16889242613581723 ['/Users/rziiiii/dev/workspace/data/raw/soxx_constituents/pit_holdings_history.csv', '/Users/rziiiii/dev/workspace/data/raw/soxx_constituents/pit_price_manifest.json', '/Users/rziiiii/dev/workspace/data/raw/soxx_leveraged_training/manifest.json']


## 结论

来源门失败；不计算因子、标签或账户。详见 conclusion.md 与 source_preflight_result.json。